In [ ]:
from pathlib import Path
from types import SimpleNamespace
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from matplotlib.colors import BoundaryNorm, ListedColormap

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REFERENCE_DIR = ROOT / "reference"
if str(REFERENCE_DIR) not in sys.path:
    sys.path.insert(0, str(REFERENCE_DIR))

from generate import generate_single_volume
from losses import setup_tpc_bins
from models import CustomVAE, DDPM, TimeUNet, TorchFEMMesh
from utils import compute_volume_tpc, plot_tpc_comparison, visualize_3d_microstructure

In [ ]:
def parse_float_targets(text):
    targets = {}
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        key, value = item.split(":", 1)
        targets[float(key)] = float(value)
    return targets


def parse_int_targets(text):
    if not text:
        return None
    return {int(k): float(v) for k, v in (item.split(":", 1) for item in text.split(","))}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device(requested):
    if requested == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = torch.device(requested)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested, but torch.cuda.is_available() is false.")
    return device


def load_models(args, device):
    vae = CustomVAE(latent_ch=args.latent_ch).to(device)
    vae_checkpoint = torch.load(args.vae_ckpt, map_location=device)
    vae.load_state_dict(vae_checkpoint["vae"] if "vae" in vae_checkpoint else vae_checkpoint)
    vae.eval()
    for param in vae.parameters():
        param.requires_grad_(False)

    unet = TimeUNet(args.latent_ch).to(device)
    unet.load_state_dict(torch.load(args.unet_ckpt, map_location=device))
    unet.eval()
    for param in unet.parameters():
        param.requires_grad_(False)
    return vae, unet


def load_tpc_targets(path):
    train_data = np.load(path)
    phases = sorted(int(key[1]) for key in train_data.files if key.endswith("_mean"))
    targets = {phase: train_data[f"S{phase}{phase}_mean"] for phase in phases}
    return phases, targets


def save_slice_preview(volume, save_path):
    z_mid = volume.shape[0] // 2
    y_mid = volume.shape[1] // 2
    x_mid = volume.shape[2] // 2
    slices = [("Z mid", volume[z_mid]), ("Y mid", volume[:, y_mid, :]), ("X mid", volume[:, :, x_mid])]
    cmap = ListedColormap(["#333333", "#9a9a9a", "#f2f2f2"])
    norm = BoundaryNorm([-0.01, 0.25, 0.75, 1.01], cmap.N)

    fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
    for ax, (title, image) in zip(axes, slices, strict=True):
        ax.imshow(image, cmap=cmap, norm=norm, interpolation="nearest")
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.savefig(save_path, dpi=160)
    plt.close(fig)

In [ ]:
args = SimpleNamespace(
    vae_ckpt=ROOT / "microlad-anode" / "vae_anode.pth",
    unet_ckpt=ROOT / "microlad-anode" / "unet_anode.pth",
    training_tpc=ROOT / "microlad-anode" / "autocorr_periodic_mean_std.npz",
    save_dir=ROOT / "output" / "reference_generate_smoke",
    device="auto",
    seed=0,
    n_volumes=1,
    ddpm_steps=25,
    sds_steps=0,
    sds_lr=0.001,
    t_min=None,
    t_max=None,
    refinement_K=0,
    num_samples=16,
    latent_ch=4,
    H=16,
    W=16,
    vf_targets="0:0.35,0.5:0.28,1:0.37",
    vf_weight=0.0,
    tpc_weight=0.0,
    rd_weight=0.0,
    sa_weight=0.0,
    rd_targets=None,
    sa_targets=None,
    skip_visualization=True,
    plot_tpc=False,
)

args.t_min = min(200, max(0, args.ddpm_steps // 5)) if args.t_min is None else args.t_min
args.t_max = min(500, args.ddpm_steps) if args.t_max is None else args.t_max
vf_targets = parse_float_targets(args.vf_targets)
args.vf0 = vf_targets.get(0.0, 0.35)
args.vf05 = vf_targets.get(0.5, 0.28)
args.vf1 = vf_targets.get(1.0, 0.37)
args.w_m1 = args.vf_weight
args.w_m2 = args.vf_weight

if args.num_samples != 16 or args.H != 16 or args.W != 16:
    raise ValueError("The reference 64x64x64 decoder path expects num_samples=H=W=16.")
if args.ddpm_steps < 1:
    raise ValueError("ddpm_steps must be at least 1.")
if args.sds_steps > 0 and not (0 <= args.t_min < args.t_max <= args.ddpm_steps):
    raise ValueError("t_min and t_max must satisfy 0 <= t_min < t_max <= ddpm_steps.")
for required in (args.vae_ckpt, args.unet_ckpt, args.training_tpc):
    if not required.exists():
        raise FileNotFoundError(required)

seed_everything(args.seed)
device = choose_device(args.device)
args.save_dir.mkdir(parents=True, exist_ok=True)

vae, unet = load_models(args, device)
ddpm = DDPM(timesteps=args.ddpm_steps, beta_start=1e-4, beta_end=2e-2, device=device)
phases, tpc_targets = load_tpc_targets(args.training_tpc)
rd_targets = parse_int_targets(args.rd_targets)
sa_targets = parse_int_targets(args.sa_targets)
fem_solver = TorchFEMMesh(M=64, N=64, low_cond=0.001, device=device).to(device) if args.rd_weight > 0 else None
bin_mat, bin_counts = setup_tpc_bins(64, 64, device) if args.sds_steps > 0 and args.tpc_weight > 0 else (None, None)

In [ ]:
for volume_index in range(args.n_volumes):
    volume = generate_single_volume(
        vae,
        unet,
        ddpm,
        fem_solver,
        bin_mat,
        bin_counts,
        tpc_targets,
        sa_targets,
        rd_targets,
        phases,
        args,
        device,
    )

    out_dir = args.save_dir / f"volume_{volume_index:03d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    volume_path = out_dir / "volume.tiff"
    preview_path = out_dir / "middle_slices.png"
    tifffile.imwrite(volume_path, volume.astype(np.float32))
    save_slice_preview(volume, preview_path)

    if not args.skip_visualization:
        visualize_3d_microstructure(volume, out_dir / "3d_visualization_final.png")
    if args.plot_tpc:
        volume_labels = (volume * 2).astype(np.int32)
        tpc_results = compute_volume_tpc(volume_labels, phases)
        for axis_name in ("x", "y", "z"):
            axis_tpcs = {phase: tpc_results[(axis_name, phase)] for phase in phases}
            plot_tpc_comparison(axis_tpcs, tpc_targets, phases, out_dir / f"tpc_compare_{axis_name}.png", axis_name)

    print(volume_path)
    print(preview_path)